In [1]:
import os
#os.environ['IVOA_REGISTRY']="http://vao.stsci.edu/RegTAP/TapService.aspx"

import pyvo as vo
import warnings
# There are a number of relatively unimportant warnings that show up, so for now, suppress them:
warnings.filterwarnings("ignore", module="astropy.nddata.blocks.*")
warnings.filterwarnings("ignore", module="pyvo.utils.xml.*")
warnings.filterwarnings("ignore", module="urllib3.connectionpool.*")

#T Dower said:  " ForRegTAP, the preferred URL is now 
#  https://mast.stsci.edu/vo-tap/api/v0.1/registry. 
#  OAI-PMH is still on the old system."
navo_new_regtap = vo.dal.TAPService('https://mast.stsci.edu/vo-tap/api/v0.1/registry')
navo_old_regtap = vo.dal.TAPService('https://vao.stsci.edu/RegTAP/TapService.aspx')
gavo_regtap = vo.dal.TAPService('https://dc.zah.uni-heidelberg.de/__system__/tap/run')
euvo_regtap = vo.dal.TAPService('https://registry.euro-vo.org/regtap/tap')

from pyvo import registry
from astropy.coordinates import SkyCoord

# Registry Spring Cleaning notebook

Following up on Markus' [Confessions of a Registry Janitor](https://blog.g-vo.org/registry-a-janitor-speaks-out.html), I propose some regular checks of the metadata.  We already have checks of the validity of services, for instance, in the Operations group weather reports.  This would be compolementary.

## Check 1:  spot check numbers between different registries

What's the best way to get the current registries?  Testing one of them seems circular.  But the [RofR](https://rofr.ivoa.net) is still pointing to the old NAVO RegTAP.  OTOH, there's a bug in the new one.  

In [2]:
result = registry.search(datamodel="regtap").to_table()
print(result['ivoid','access_urls'])

             ivoid              ...
------------------------------- ...
         ivo://aip.gavo.org/tap ...
 ivo://archive.stsci.edu/regtap ...
       ivo://esavo/registry/tap ...
               ivo://fai.kz/tap ...
          ivo://org.gavo.dc/tap ...
ivo://purx/lamost/dr10/v2.0/tap ...
ivo://purx/lamost/dr11/v2.0/tap ...
                 ivo://purx/tap ...


In [3]:
def compare( query ):
    # Currently 
    #navo_regtap = vo.dal.TAPService('https://vao.stsci.edu/RegTAP/TapService.aspx')
    #navo_new_regtap = vo.dal.TAPService('https://mast.stsci.edu/vo-tap/api/v0.1/registry')
    navo_regtap = vo.dal.TAPService('https://mast.stsci.edu/vo-tap/api/v0.1/registry')
    gavo_regtap = vo.dal.TAPService('https://dc.zah.uni-heidelberg.de/__system__/tap/run')
    euvo_regtap = vo.dal.TAPService('https://registry.euro-vo.org/regtap/tap')
    padc_regtap = vo.dal.TAPService('http://voparis-rr.obspm.fr/tap')
    sets=[]
    for name,regtap in [('NAVO',navo_regtap),('GAVO',gavo_regtap),("EUVO",euvo_regtap),("PADC",padc_regtap)]:
        try:
            sias = regtap.search(query)
            print(f"{name} RegTAP finds {len(sias)}")
            #strings=sias.to_table()['ivoid'] + ' "'+ sias.to_table()['cap_description']+'"'
            sets.append( set(sias.to_table()['ivoid'].data) )
        except Exception as e:
            print(f"{name} RegTAP gives error: {e}")
            
    print("Unique to NAVO:")
    print(sets[0].difference(sets[1],sets[2],sets[3]))
    print("Unique to GAVO:")
    print(sets[1].difference(sets[0],sets[2],sets[3]))
    print("Unique to EUVO:")
    print(sets[2].difference(sets[0],sets[1],sets[3]))
    print("Unique to PADC:")
    print(sets[3].difference(sets[0],sets[1],sets[2]))

    print("Missing from NAVO but in all the others:")
    print( (sets[1] & sets[2] & sets[3]) - sets[0] )
    print("Missing from GAVO but in all the others:")
    print( (sets[0] & sets[2] & sets[3]) - sets[1] )
    print("Missing from EUVO but in all the others:")
    print( (sets[0] & sets[1] & sets[3]) - sets[2] )
    print("Missing from PADC but in all the others:")
    print( (sets[0] & sets[1] & sets[2]) - sets[3] )

    print("Anything not in all 4")
    print( (sets[0] | sets[1] | sets[2] | sets[3]) - (sets[0] & sets[1] & sets[2] & sets[3]) )

    return(sets)

In [4]:
sets = compare("select * from rr.capability where standard_id like 'ivo://ivoa.net/std/sia%'")

/home/sboomi/ivoa/registry-housekeeping/.venv/lib/python3.13/site-packages/pyvo/dal/query.py:403: DALOverflowWarning: Results truncated due to server limits. Consider setting a maxrec value.
  warn("Results truncated due to server limits. Consider "


NAVO RegTAP finds 493
GAVO RegTAP finds 491
EUVO RegTAP finds 492
PADC RegTAP finds 491
Unique to NAVO:
{'ivo://padc.obspm.astro/esor/q/i', 'ivo://padc.obspm.astro/posse/q/siap', 'ivo://irsa.ipac/spitzer/images/level1', 'ivo://padc.obspm.astro/posse/q/cutout', 'ivo://padc.obspm.astro/srcj/q/i', 'ivo://vopdc.obspm/dfbs', 'ivo://padc.obspm.astro/dfbs/q/i', 'ivo://irsa.ipac/spitzer/images/level2', 'ivo://irsa.ipac/mast/scrapbook', 'ivo://irsa.ipac/dss/images', 'ivo://org.gavo.dc/maidanak/res/rawframes/rawframes', 'ivo://nasa.heasarc/skyview/planck030'}
Unique to GAVO:
set()
Unique to EUVO:
set()
Unique to PADC:
set()
Missing from NAVO but in all the others:
{'ivo://fai.kz/maksutov_50_telescope/q/i', 'ivo://padc.obspm.astro-m/srcj/q/i', 'ivo://astro.upjs/upjs_img/i/i', 'ivo://astro.upjs/__system__/siap2/sitewide', 'ivo://padc.obspm.astro-m/posse/q/siap', 'ivo://padc.obspm.astro-m/dfbs/q/i', 'ivo://padc.obspm.astro-m/posse/q/cutout', 'ivo://fai.kz/schmidt_telescope_lc/q/i', 'ivo://iucaa/crt

In [5]:
sets = compare("select * from rr.capability where standard_id like 'ivo://ivoa.net/std/hips%'")

NAVO RegTAP finds 533
GAVO RegTAP finds 687
EUVO RegTAP finds 687
PADC RegTAP finds 687
Unique to NAVO:
set()
Unique to GAVO:
set()
Unique to EUVO:
set()
Unique to PADC:
set()
Missing from NAVO but in all the others:
{'ivo://cds/p/quest-for-the-missing-dust/pacs100', 'ivo://cds/p/dm/simbad-biblio/pub-dates/1960-1965', 'ivo://cds/p/dm/simbad-biblio/pub-dates/1994', 'ivo://cds/p/galaxycounts/2mpz/003-004', 'ivo://cds/p/mercury/dem-665m', 'ivo://cds/p/dm/simbad-biblio/pub-dates/2000', 'ivo://cds/p/europa/voyager-galileossi-500m', 'ivo://cds/p/dm/simbad-biblio/otypes/hii', 'ivo://cds/p/jwst/carina-nebula/nircam', 'ivo://cds/p/meerkat/galactic-centre-spectral-index', 'ivo://cds/p/dm/simbad-biblio/pub-dates/1965-1970', 'ivo://cds/p/venus/magellan/meterscaleslope-4641m', 'ivo://cds/p/skymapper/dr4/i', 'ivo://cds/p/mars/themis-ir-night-100m-v14', 'ivo://cds/p/dm/flux-color-rp-g-bp/i/345/gaia2', 'ivo://cds/p/act-planck/dr4dr6/f090/pointsource', 'ivo://cds/c/dm/simbad-biblio/pub-dates/1970-now',

In [6]:
sets = compare("select * from rr.capability where standard_id like 'ivo://ivoa.net/std/cone%' and ivoid not like '%vizier%'")

NAVO RegTAP finds 2038
GAVO RegTAP finds 2001
EUVO RegTAP finds 2029
PADC RegTAP finds 2001
Unique to NAVO:
{'ivo://xaovo/wugang/wugang/q', 'ivo://xaovo/pulsar/pulsar/q', 'ivo://sao.ru/dsa-cats/wsdb', 'ivo://au.csiro/psrda/atnf_pulsar_scs', 'ivo://xaovo/nsone/q/web', 'ivo://xaovo/pul/pulsar/q', 'ivo://astron.nl/hetdex/lotss-dr1-raw/cone', 'ivo://xaovo/liujun/liujun/q', 'ivo://ads.harvard.edu/cone'}
Unique to GAVO:
set()
Unique to EUVO:
{'ivo://astronet.ru/cas/wise'}
Unique to PADC:
set()
Missing from NAVO but in all the others:
{'ivo://kasi_vo/nsvs/cs', 'ivo://astro.upjs/gaiadr3_eb/t/gdr3cone', 'ivo://astro.upjs/upjs_ts/t/kolonica-objects', 'ivo://astro.upjs/upjs_gaia_eb/q/upjs_eb_cone', 'ivo://astro.upjs/personal/t/personal-objects', 'ivo://astro.upjs/ogle/o/ogle-objects', 'ivo://wfau.roe.ac.uk/glimpse-dsa'}
Missing from GAVO but in all the others:
set()
Missing from EUVO but in all the others:
set()
Missing from PADC but in all the others:
set()
Anything not in all 4
{'ivo://astronet

The hard part is then looking at those and understanding why.  What other information would we want to look at?

## Check 2:  UCDs 

#### Look at all UCDs in the Registry

In [7]:
from astropy.io.votable.ucd import check_ucd
# add: having ucd is not null afger group by ucd to remove nulls

query="""
  select distinct ucd, count(*) as cnt
  from rr.table_column 
  group by ucd 
  order by cnt desc
  
  """
result = gavo_regtap.search(query)

#print(result)

all_ucds = result.to_table()
invalid_ucds = []
for i,u in enumerate(all_ucds['ucd'].data):
    if not check_ucd(u):
        invalid_ucds.append((u,all_ucds['cnt'][i]))
print(f"Found {len(invalid_ucds)} invalid or unspecified UCDs")
print(f"  The top 10 bad UCD values by number of instances are")
x=[print(f"{c[0]:25}: {c[1]}") for c in invalid_ucds[0:10] ]

Found 162 invalid or unspecified UCDs
  The top 10 bad UCD values by number of instances are
                         : 236458
??                       : 30342
phot.flux.density;       : 672
meta.code.qual,stat.fit  : 237
vox:image_filesize       : 129
????                     : 70
image?                   : 49
phot.mag;                : 42
vox:image_mjdateobs      : 42
vox:bandpass_hilimit     : 40


Note that the numbers of "??" and "????" have not changed since [Markus' post in 2023](https://blog.g-vo.org/registry-a-janitor-speaks-out.html)

In [8]:
invalid_ucds_cv = []
for i,u in enumerate(all_ucds['ucd'].data):
    if not check_ucd(u,check_controlled_vocabulary=True):
        invalid_ucds_cv.append((u,all_ucds['cnt'][i]))
print(f"Found {len(invalid_ucds_cv)} that are not valid under UCD1+ controlled vocabulary")
print(f"  The top 10 bad UCD values by number of instances are")
[print(f"{c[0]:25}: {c[1]}") for c in invalid_ucds_cv[0:10] ]

Found 1673 that are not valid under UCD1+ controlled vocabulary
  The top 10 bad UCD values by number of instances are
                         : 236458
??                       : 30342
error                    : 13825
code_misc                : 8538
phot_mag                 : 6291
obs.field                : 4531
fit_param                : 4505
number                   : 3090
id_number                : 2694
phot_intensity_adu       : 2512


[None, None, None, None, None, None, None, None, None, None]

#### UCDs at different publishers

Getting the publishers with the most resources in the Registry excluding Vizier.  Let's check those.  

In [9]:
publishers = gavo_regtap.search("""
    select distinct role_ivoid, count(*) as cnt , role_name
    from rr.res_role 
    where base_role = 'publisher' and role_name != 'CDS'
    group by role_ivoid, role_name
    order by cnt desc
    """).to_table()[0:10]
publishers



role_ivoid,cnt,role_name
object,int32,object
ivo://nasa.heasarc/asd,1094,NASA/GSFC HEASARC
ivo://irsa.ipac/irsa,628,NASA/IPAC Infrared Science Archive
,253,The GAVO DC team
,222,Planetary Data System
ivo://wfau.roe.ac.uk,122,"WFAU, Institute for Astronomy, University of Edinburgh"
ivo://archive.stsci.edu/stsci-arc,101,Space Telescope Science Institute Archive
ivo://svo.cab,68,SVO CAB
ivo://noirlab.edu,65,NSF NOIRLab Astro Data Lab Team
,60,Paris Astronomical Data Centre


Adding from in person attendees:

In [10]:
from astropy.table import Table, vstack

inperson = """\
ESO, PDS, AAS, NED, China-VO, INFN, \
CfA, CXC, Rubin, INAF, Paris Astronomical Data Centre, \
Hyderabad, UCLA \
""".split(', ')

for institute in inperson:
    query=f""" 
    select distinct role_ivoid, count(*) as cnt , role_name
    from rr.res_role 
    where base_role = 'publisher' and role_name != 'CDS'
    and ( role_ivoid ilike '%{institute}%' or role_name ilike '%{institute}%' )
    group by role_ivoid, role_name
    order by cnt desc
    """
    r=gavo_regtap.search(query).to_table()
    #print(r)
    if len(r) > 0:
        publishers=vstack([publishers, r[0]])
publishers


role_ivoid,cnt,role_name
object,int32,object
ivo://nasa.heasarc/asd,1094,NASA/GSFC HEASARC
ivo://irsa.ipac/irsa,628,NASA/IPAC Infrared Science Archive
,253,The GAVO DC team
,222,Planetary Data System
ivo://wfau.roe.ac.uk,122,"WFAU, Institute for Astronomy, University of Edinburgh"
ivo://archive.stsci.edu/stsci-arc,101,Space Telescope Science Institute Archive
ivo://svo.cab,68,SVO CAB
ivo://noirlab.edu,65,NSF NOIRLab Astro Data Lab Team
,60,Paris Astronomical Data Centre


In [11]:
## Helper function to grab metadata, group it by publisher, 
##   and look for invalid values, print a summary
def validate_publishers(query, publist, badvallist, label, quiet=False):
    import pandas as pd #  Handy functions
    warnings.filterwarnings("ignore", message=".*This pattern is interpreted as a regular expression.*")
    for i,p in enumerate([pp.strip() for pp in publist['role_name'].data]):
        #  Look at all the metadata from this publisher
        if not quiet: print(f"\nlooking at publisher {p}")
        try: 
            results = gavo_regtap.search(query.replace("xxxx",p))
        except Exception as e:
            print(f"    Encountered exception {e} during query on publisher {p}")
            continue
        if len(results) != 0 and not quiet:  
            print(f"    publisher {p} publishes {len(results)} distinct values of {label}")
        elif not quiet: 
            print(f"    publisher {p} publishes no such metadata (?)")
            continue #  ?

        ##  
        df = pd.DataFrame(data={
            label:results.to_table()[label].data.data,
            "cnt":results.to_table()['cnt'].data.data
        })
        pcount = 0
        for c in badvallist:  #  invalid_ucds or invalid_ucds_cv (this is huge)
            #  c is a tuple of the string and the count
            if c[0]=='':  
                matches = df[label].astype(str).str.len() == 0
            elif '?' in c[0]:
                matches = df[label].str.contains("?",regex=False)
            else:
                matches = df[label] == c[0]
            cnt = df[matches]['cnt'].sum() # should only be one 
            if cnt == 0:  
                continue
            print(f"    value '{c[0]}' used {cnt} times")
            pcount += 1
            if pcount > 10:  break

In [12]:
query = f"""
        select ucd, count(*) as cnt from ( rr.res_role natural join rr.table_column )
        where role_name = 'xxxx'
        group by ucd 
        """
validate_publishers( query, publishers, invalid_ucds, "ucd")


looking at publisher NASA/GSFC HEASARC
    publisher NASA/GSFC HEASARC publishes 2481 distinct values of ucd
    value '' used 8037 times

looking at publisher NASA/IPAC Infrared Science Archive
    publisher NASA/IPAC Infrared Science Archive publishes 1 distinct values of ucd
    value '' used 124 times

looking at publisher The GAVO DC team
    publisher The GAVO DC team publishes 857 distinct values of ucd
    value '' used 1525 times
    value 'vox:image_filesize' used 46 times
    value 'vox:image_mjdateobs' used 2 times

looking at publisher Planetary Data System
    publisher Planetary Data System publishes 41 distinct values of ucd
    value '' used 2664 times

looking at publisher WFAU, Institute for Astronomy, University of Edinburgh
    publisher WFAU, Institute for Astronomy, University of Edinburgh publishes 855 distinct values of ucd
    value '' used 301495 times
    value '??' used 60549 times
    value '????' used 60549 times
    value 'image?' used 60549 times
    v

In [13]:
query = f"""
        select ucd, count(*) as cnt from ( rr.res_role natural join rr.table_column )
        where role_name = 'xxxx'
        group by ucd 
        """
validate_publishers( query, publishers, invalid_ucds_cv, "ucd")


looking at publisher NASA/GSFC HEASARC
    publisher NASA/GSFC HEASARC publishes 2481 distinct values of ucd
    value '' used 8037 times

looking at publisher NASA/IPAC Infrared Science Archive
    publisher NASA/IPAC Infrared Science Archive publishes 1 distinct values of ucd
    value '' used 124 times

looking at publisher The GAVO DC team
    publisher The GAVO DC team publishes 857 distinct values of ucd
    value '' used 1525 times
    value 'vox:image_filesize' used 46 times
    value 'vox:image_mjdateobs' used 2 times

looking at publisher Planetary Data System
    publisher Planetary Data System publishes 41 distinct values of ucd
    value '' used 2664 times

looking at publisher WFAU, Institute for Astronomy, University of Edinburgh
    publisher WFAU, Institute for Astronomy, University of Edinburgh publishes 855 distinct values of ucd
    value '' used 301495 times
    value '??' used 60549 times
    value 'error' used 25260 times
    value 'code_misc' used 14690 times
 

In [14]:
culprits = []
for i,u in enumerate(all_ucds['ucd'].data):
    if not check_ucd(u,check_controlled_vocabulary=True):
        culprits.append((u,all_ucds['cnt'][i]))
print(f"Found {len(culprits)} that are not valid under UCD1+ controlled vocabulary")
print(f"  The top 10 bad UCD values by number of instances are")
x=[print(f"{c[0]:25}: {c[1]}") for c in culprits[0:10] ]

Found 1673 that are not valid under UCD1+ controlled vocabulary
  The top 10 bad UCD values by number of instances are
                         : 236458
??                       : 30342
error                    : 13825
code_misc                : 8538
phot_mag                 : 6291
obs.field                : 4531
fit_param                : 4505
number                   : 3090
id_number                : 2694
phot_intensity_adu       : 2512


## Check 3:  authors

In [15]:
query = f"""
    select distinct role_name, count(*) as cnt 
    from rr.res_role 
    where base_role = 'creator' 
    group by role_name
    """
gavo_regtap.search(query).to_table()

/home/sboomi/ivoa/registry-housekeeping/.venv/lib/python3.13/site-packages/pyvo/dal/query.py:403: DALOverflowWarning: Results truncated due to server limits. Consider setting a maxrec value.
  warn("Results truncated due to server limits. Consider "


role_name,cnt
object,int32
"Guo W.-J.,Zhang Z.-X.",1
"Oh K.,Rosario D.J.",1
YuL.,1
"Santos-Sanz P.,Wilson T.G.",1
Holliman M.J.,1
"Lagrange A.-M.,Langlois M.",1
"Burstein D.,Bohlin R.C.",1
DAVILA H.,1
"Dumusque X.,Fulton B.J.",1


Have
* Last F.
* Last F., Last2 F.
* Last, F.
* F. Last, Last2. F.

At least where there are commas they are used to separate two authors, rather than "Last, F" or something.

In [16]:
names = gavo_regtap.search("select distinct role_name, count(*) as cnt from rr.res_role where base_role = 'creator' group by role_name").to_table()
names

role_name,cnt
object,int32
"Guo W.-J.,Zhang Z.-X.",1
"Oh K.,Rosario D.J.",1
YuL.,1
"Santos-Sanz P.,Wilson T.G.",1
Holliman M.J.,1
"Lagrange A.-M.,Langlois M.",1
"Burstein D.,Bohlin R.C.",1
DAVILA H.,1
"Dumusque X.,Fulton B.J.",1


## Check 4:  subjects and the UAT

In [17]:
subjects = gavo_regtap.search("select res_subject, count(*) as cnt from rr.res_subject group by res_subject order by cnt desc").to_table()
subjects

res_subject,cnt
object,int32
visible-astronomy,8027
spectroscopy,4661
galaxies,4572
infrared-photometry,4490
photometry,4144
radial-velocity,3025
surveys,2938
redshifted,2831
variable-stars,2142


In [18]:
import urllib.request, json 
with urllib.request.urlopen("https://raw.githubusercontent.com/astrothesaurus/UAT/master/UAT.json") as url:
    uat = json.load(url)

In [19]:
#  Generator that goes through the nested JSON and looks for a key anywhere down in it
def item_generator(json_input, lookup_key):
    if isinstance(json_input, dict):
        for k, v in json_input.items():
            if k == lookup_key:
                yield v
            else:
                yield from item_generator(v, lookup_key)
    elif isinstance(json_input, list):
        for item in json_input:
            yield from item_generator(item, lookup_key)

In [20]:
uat_name_list = [x.lower() for x in item_generator(uat,'name')]
print(f"Found {len(uat_name_list)} names in the UAT")
print(uat_name_list[0:10])

Found 4559 names in the UAT
['astrophysical processes', 'astrophysical magnetism', 'cosmic magnetic fields theory', 'emerging flux tubes', 'magnetic fields', 'geomagnetic fields', 'magnetic anomalies', 'primordial magnetic fields', 'gravitation', 'relativity']


In [21]:
invalid_subjects = []
correct_subjects = []
for i,s in enumerate(subjects['res_subject'].data):
    if s.lower() in uat_name_list:
        correct_subjects.append((s,subjects['cnt'][i]))
    else:
        invalid_subjects.append((s,subjects['cnt'][i]))
print(f"Found {len(invalid_subjects)} Registry res_subject entries \
that are not in the UAT and {len(correct_subjects)} that are.")
print(f"  The top 10 bad subject values by number of instances are")
x=[print(f"{c[0]}: {c[1]}") for c in invalid_subjects[0:10] ]

Found 1004 Registry res_subject entries that are not in the UAT and 234 that are.
  The top 10 bad subject values by number of instances are
visible-astronomy: 8027
infrared-photometry: 4490
radial-velocity: 3025
variable-stars: 2142
multiple-stars: 1998
x-ray-sources: 1809
Wide-band photometry: 1809
open-star-clusters: 1752
chemical-abundances: 1725
active-galactic-nuclei: 1502


In [22]:
import re
result = [u for u in uat_name_list if re.search("^star.*",u)]
print(f"Found {len(result)} matches to 'star' such as")
print(result[0:10])

Found 18 matches to 'star' such as
['star-planet interactions', 'starburst galaxies', 'starburst galaxies', 'starburst galaxies', 'starspots', 'starspots', 'star-planet interactions', 'star atlases', 'star counts', 'star counts']


In [23]:
query = f"""
        select top 10 res_subject, count(*) as cnt from ( rr.res_role natural join rr.res_subject )
        where role_name = 'xxxx'
        group by res_subject order by cnt desc
        """
validate_publishers( query, publishers, invalid_subjects, "res_subject")


looking at publisher NASA/GSFC HEASARC
    publisher NASA/GSFC HEASARC publishes 10 distinct values of res_subject
    value 'Survey Source' used 654 times
    value 'Observation' used 87 times
    value 'Star' used 70 times
    value 'Galaxy' used 25 times
    value 'GRB' used 31 times
    value 'AGN' used 22 times
    value 'Cluster of Galaxies' used 15 times
    value 'XRB' used 11 times
    value 'Optical Counterpart' used 11 times

looking at publisher NASA/IPAC Infrared Science Archive
    publisher NASA/IPAC Infrared Science Archive publishes 10 distinct values of res_subject
    value '' used 183 times
    value 'extragalactic survey' used 101 times
    value 'all sky survey' used 58 times
    value 'survey' used 24 times
    value 'high redshift galaxies' used 17 times

looking at publisher The GAVO DC team
    publisher The GAVO DC team publishes 10 distinct values of res_subject
    value 'proper-motions' used 31 times
    value 'milky-way-galaxy' used 18 times
    value 'v

## Check 5:  concepts

In [24]:
reg_uat_concept_list = gavo_regtap.search("select distinct uat_concept from rr.subject_uat").to_table()["uat_concept"].data
print(f"There are {len(reg_uat_concept_list)} distinct uat_concept values in the registry's subject_uat table")

There are 475 distinct uat_concept values in the registry's subject_uat table


In [25]:
bad=[]
for c in reg_uat_concept_list:
    # lower case and replace - with space
    if c.lower().replace("-"," ") not in uat_name_list:
        bad.append(c)
print(f"There are {len(bad)} concepts not found in the UAT such as:")
print(bad[0:10])

There are 47 concepts not found in the UAT such as:
['active-galactic-nuclei ', 'astrl', 'astronomical-simulations ', 'early-type-galaxies', 'early-type-stars', 'earth-planet', 'exoplanet-atmospheric-composition', 'gamma-ray-astronomy', 'gamma-ray-bursts', 'gamma-ray-bursts ']


In [26]:
query = f"""
        select top 10 uat_concept, count(*) as cnt from ( rr.res_role natural join rr.subject_uat )
        where role_name = 'xxxx'
        group by uat_concept order by cnt desc
        """
validate_publishers( query, publishers, bad, "uat_concept")


looking at publisher NASA/GSFC HEASARC
    publisher NASA/GSFC HEASARC publishes 10 distinct values of uat_concept

looking at publisher NASA/IPAC Infrared Science Archive
    publisher NASA/IPAC Infrared Science Archive publishes 10 distinct values of uat_concept

looking at publisher The GAVO DC team
    publisher The GAVO DC team publishes 10 distinct values of uat_concept

looking at publisher Planetary Data System
    publisher Planetary Data System publishes 6 distinct values of uat_concept

looking at publisher WFAU, Institute for Astronomy, University of Edinburgh
    publisher WFAU, Institute for Astronomy, University of Edinburgh publishes 7 distinct values of uat_concept

looking at publisher Space Telescope Science Institute Archive
    publisher Space Telescope Science Institute Archive publishes 10 distinct values of uat_concept

looking at publisher SVO CAB
    publisher SVO CAB publishes 6 distinct values of uat_concept

looking at publisher NSF NOIRLab Astro Data Lab 

## Check 6: Spatial coverage

Spatial coverage enables registry-wide spatial searches.  But HEASARC for example specifies full sky coverage for all of its services even when the data are not full sky but a sample distributed across the full sky.   

In [27]:
query = f"""
        select distinct role_name, count(*) as cnt 
        from ( rr.res_role natural join rr.stc_spatial )
        where base_role = 'publisher' and ( coverage = '' or coverage = '0/0-11' )
        group by role_name 
        order by cnt desc
        """
gavo_regtap.search(query).to_table()

role_name,cnt
object,int32
NASA/GSFC HEASARC,1013
CDS,63
The GAVO DC team,39
\nChandra X-ray Observatory\n,8
ASTRON,8
BSDC,7
NASA/IPAC Infrared Science Archive,6
GCN,5
Canadian Astronomy Data Centre,4


## Check 7: Relationships

This is an outstanding discussion I believe, so this is not necessarily wrong, depending on who you ask.  ;) 

In [28]:
gavo_regtap.search("""
    select distinct role_name, count(*) as cnt
    from ( rr.relationship natural join rr.res_role )
    where relationship_type = 'related-to' and base_role = 'publisher'
    group by role_name
    order by cnt desc
""").to_table()

role_name,cnt
object,int32
CDS,174942
"WFAU, Institute for Astronomy, University of Edinburgh",51
\n International Virtual Observatory Alliance\n,12
The China-VO Team,8
CSIRO,7
IDOC D2S,7
IDOC GINCO,6
LTE - Paris Astronomical Data Centre,5
Mullard Space Science Laboratory,4


## To be expanded.  Now what to do with this?  

* Report cross-checks between registries to their admins.  
* Compile a report of issues as above and advertise at IVOA Interop's Registry (or Ops?) session.  
* Compile a report of issues found for each publisher and email them yearly to request updates.  


## Scratch 